# Diabetes Big Data Pipeline:  Apache Spark & MongoDB

### Group 11: Prayusha Poudel, Tuo Yan , Dan Le 

## Section 1: Setup & Imports

In [1]:
# Install required packages
# Run this cell once, then restart the kernel before continuing
%pip install pyspark pymongo pandas matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import pymongo
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, desc, rank, when
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)

print("All imports successful!")

All imports successful!


## Section 2: Initialize SparkSession (with MongoDB Connector)


In [3]:
# Create SparkSession with MongoDB connector
spark = SparkSession.builder \
    .appName("DiabetesBigDataPipeline") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0") \
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017/diabetes_db.patients") \
    .config("spark.mongodb.write.connection.uri", "mongodb://localhost:27017/diabetes_db.patients") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession created successfully!")
print(f"Spark version: {spark.version}")

SparkSession created successfully!
Spark version: 4.1.1


## Section 3: Ingest CSV Data into MongoDB

In [4]:
# Connect to local MongoDB
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["diabetes_db"]
collection = db["patients"]

# Load CSV with pandas
df_csv = pd.read_csv("diabetes_dataset.csv")

print(f"CSV loaded: {len(df_csv)} rows, {len(df_csv.columns)} columns")
print("Columns:", list(df_csv.columns))
df_csv.head()

CSV loaded: 100000 rows, 16 columns
Columns: ['year', 'gender', 'age', 'location', 'race:AfricanAmerican', 'race:Asian', 'race:Caucasian', 'race:Hispanic', 'race:Other', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'hbA1c_level', 'blood_glucose_level', 'diabetes']


,year,gender,age,location,race:AfricanAmerican,race:Asian,race:Caucasian,race:Hispanic,race:Other,hypertension,heart_disease,smoking_history,bmi,hbA1c_level,blood_glucose_level,diabetes
0,2020,Female,32.0,Alabama,0,0,0,0,1,0,0,never,27.32,5.0,100,0
1,2015,Female,29.0,Alabama,0,1,0,0,0,0,0,never,19.95,5.0,90,0
2,2015,Male,18.0,Alabama,0,0,0,0,1,0,0,never,23.76,4.8,160,0
3,2015,Male,41.0,Alabama,0,0,1,0,0,0,0,never,27.32,4.0,159,0
4,2016,Female,52.0,Alabama,1,0,0,0,0,0,0,never,23.75,6.5,90,0


In [5]:
# Insert into MongoDB
# Clear old data first to avoid duplicates on re-runs
collection.delete_many({})

# Convert to list of dicts and insert
records = df_csv.to_dict(orient="records")
collection.insert_many(records)

print(f"Successfully ingested {len(records)} records into MongoDB.")
print(f"Verification — documents in collection: {collection.count_documents({})}")

Successfully ingested 100000 records into MongoDB.
Verification — documents in collection: 100000


## Section 4: Read from MongoDB → Spark DataFrame

In [6]:
# Step 1: Fetch data from MongoDB using PyMongo
mongo_data = list(collection.find())

# Step 2: Remove the '_id' field — ObjectId cannot be serialized by Spark
mongo_data = [{k: v for k, v in doc.items() if k != "_id"} for doc in mongo_data]

print(f"Fetched {len(mongo_data)} documents from MongoDB")
print("Sample document keys:", list(mongo_data[0].keys()) if mongo_data else "No data")

Fetched 100000 documents from MongoDB
Sample document keys: ['year', 'gender', 'age', 'location', 'race:AfricanAmerican', 'race:Asian', 'race:Caucasian', 'race:Hispanic', 'race:Other', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'hbA1c_level', 'blood_glucose_level', 'diabetes']


In [7]:
# Step 3: Create Spark DataFrame
# We let Spark infer the schema first to see what columns exist
df_raw = spark.createDataFrame(mongo_data)
df_raw.printSchema()
df_raw.show(3)

root
 |-- age: double (nullable = true)
 |-- blood_glucose_level: long (nullable = true)
 |-- bmi: double (nullable = true)
 |-- diabetes: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- hbA1c_level: double (nullable = true)
 |-- heart_disease: long (nullable = true)
 |-- hypertension: long (nullable = true)
 |-- location: string (nullable = true)
 |-- race:AfricanAmerican: long (nullable = true)
 |-- race:Asian: long (nullable = true)
 |-- race:Caucasian: long (nullable = true)
 |-- race:Hispanic: long (nullable = true)
 |-- race:Other: long (nullable = true)
 |-- smoking_history: string (nullable = true)
 |-- year: long (nullable = true)

+----+-------------------+-----+--------+------+-----------+-------------+------------+--------+--------------------+----------+--------------+-------------+----------+---------------+----+
| age|blood_glucose_level|  bmi|diabetes|gender|hbA1c_level|heart_disease|hypertension|location|race:AfricanAmerican|race:Asian|race:Caucasian|

## Section 5: Data Cleaning

In [8]:
# Clean: cast columns to correct types and remove nulls
# Adjust column names below to match your actual diabetes_dataset.csv columns

df_clean = df_raw \
    .filter(col("age").isNotNull()) \
    .filter(col("bmi").isNotNull()) \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("bmi", col("bmi").cast("double")) \
    .withColumn("diabetes", col("diabetes").cast("int")) \
    .withColumn("blood_glucose_level", col("blood_glucose_level").cast("double"))

# Show cleaned data statistics
total = df_clean.count()
diabetic = df_clean.filter(col("diabetes") == 1).count()

print(f"Total clean records : {total}")
print(f"Diabetic patients   : {diabetic}  ({diabetic/total*100:.1f}%)")
print(f"Non-diabetic        : {total - diabetic}  ({(total-diabetic)/total*100:.1f}%)")

df_clean.printSchema()
df_clean.show(5)

Total clean records : 100000
Diabetic patients   : 8500  (8.5%)
Non-diabetic        : 91500  (91.5%)
root
 |-- age: integer (nullable = true)
 |-- blood_glucose_level: double (nullable = true)
 |-- bmi: double (nullable = true)
 |-- diabetes: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- hbA1c_level: double (nullable = true)
 |-- heart_disease: long (nullable = true)
 |-- hypertension: long (nullable = true)
 |-- location: string (nullable = true)
 |-- race:AfricanAmerican: long (nullable = true)
 |-- race:Asian: long (nullable = true)
 |-- race:Caucasian: long (nullable = true)
 |-- race:Hispanic: long (nullable = true)
 |-- race:Other: long (nullable = true)
 |-- smoking_history: string (nullable = true)
 |-- year: long (nullable = true)

+---+-------------------+-----+--------+------+-----------+-------------+------------+--------+--------------------+----------+--------------+-------------+----------+---------------+----+
|age|blood_glucose_level|  bmi|diabet